# Vessel Finder & Anomaly Detection — Syria (REAL Global Fishing Watch data)

End-to-end pipeline that pulls **real, recent** vessel-tracking data for the
**Syrian EEZ** from the [Global Fishing Watch (GFW) API](https://globalfishingwatch.org/our-apis/),
engineers per-vessel behavioural features, and flags **anomalous vessels** that
may indicate illegal, unreported & unregulated (IUU) fishing or other
suspicious activity (AIS "dark" gaps, at-sea encounters / transshipment,
heavy loitering, unusual event mixes).

> **Real data only — there is no demo/synthetic mode.** A valid GFW API token
> and network access to `globalfishingwatch.org` are **required**. If either is
> missing the notebook stops with a clear message instead of inventing data.

**Outputs written to `./outputs/`:** `syria_vessel_risk.csv`,
`syria_dashboard.html` (interactive, layer-filterable), `syria_map.png` (static).

## 1. Setup & dependencies

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib seaborn folium requests geopy python-dotenv

import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid")
OUTDIR = Path("outputs"); OUTDIR.mkdir(exist_ok=True)
print("Environment ready. Outputs ->", OUTDIR.resolve())

## 2. Your Global Fishing Watch token  *(required)*

Supply your token any of three ways (checked in order): paste it below, set
`GFW_API_TOKEN` in a `.env` file / environment variable, or type it at the
secure prompt. Get a free non-commercial token at
<https://globalfishingwatch.org/our-apis/>.

In [ ]:
import os, requests, json

try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass

PASTE_YOUR_TOKEN_HERE = ""   # <-- optional: paste your GFW token between the quotes

GFW_API_TOKEN = PASTE_YOUR_TOKEN_HERE or os.environ.get("GFW_API_TOKEN", "")
# Clean common copy/paste damage (wrapping quotes, spaces, line-wrapped JWTs).
GFW_API_TOKEN = (GFW_API_TOKEN.strip().strip('"').strip("'")
                 .replace("\n", "").replace("\r", "").replace(" ", ""))
if not GFW_API_TOKEN:
    try:
        import getpass
        GFW_API_TOKEN = getpass.getpass("Paste your GFW API token: ").strip()
    except Exception:
        GFW_API_TOKEN = ""

# REQUIRED — no demo fallback.
if not GFW_API_TOKEN:
    raise RuntimeError(
        "No GFW API token provided. This notebook runs on REAL Global Fishing Watch "
        "data only — there is no demo mode.\n"
        "Get a free token at https://globalfishingwatch.org/our-apis/ and paste it "
        "above, set GFW_API_TOKEN in .env, or type it at the prompt, then re-run."
    )

GFW_BASE = "https://gateway.api.globalfishingwatch.org/v3"
HEADERS = {"Authorization": f"Bearer {GFW_API_TOKEN}"}
print(f"Token loaded (length={len(GFW_API_TOKEN)}, ends '...{GFW_API_TOKEN[-6:]}').")

## 3. API helpers & connection check

If the token is invalid or the network blocks GFW, the notebook stops here — it does **not** silently fall back to fake data.

In [ ]:
def _raise_with_body(r):
    """raise_for_status(), but surface the API's error BODY (the real reason)."""
    try:
        r.raise_for_status()
    except requests.HTTPError as e:
        raise requests.HTTPError(f"{e} | response body: {r.text[:500]}") from None
    return r

def gfw_get(path, params=None):
    r = requests.get(f"{GFW_BASE}{path}", headers=HEADERS, params=params, timeout=90)
    return _raise_with_body(r).json()

def gfw_post(path, body, params=None):
    headers = {**HEADERS, "Content-Type": "application/json"}
    r = requests.post(f"{GFW_BASE}{path}", headers=headers, params=params, json=body, timeout=120)
    return _raise_with_body(r).json()

try:
    _demo = gfw_get("/vessels/search",
                    {"where": "flag = 'SYR'",
                     "datasets[0]": "public-global-vessel-identity:latest", "limit": 1})
    print(f"GFW connection OK — registry reports {_demo.get('total','?')} SYR-flagged vessels.")
except Exception as e:
    raise RuntimeError(
        "Could not reach the Global Fishing Watch API. This notebook uses REAL data "
        "only — no demo fallback.\n"
        f"  Reason: {e}\n"
        "  • 'Host not in allowlist' -> your network blocks globalfishingwatch.org; "
        "run on open internet or widen the egress allowlist.\n"
        "  • 401/403 invalid token   -> get/re-copy a valid token (one line, no quotes) "
        "from https://globalfishingwatch.org/our-apis/."
    ) from None

## 4. Area of interest & **recent** time window

The window is computed **relative to today** (not a hard-coded year). Set `LOOKBACK_DAYS` to `7` for the last week, `30` for the last month, `90` for the last quarter.

In [ ]:
from datetime import datetime, timezone, timedelta

SYRIA_FLAG = "SYR"
SYR_BBOX = {"lat_min": 34.55, "lat_max": 35.95, "lon_min": 33.50, "lon_max": 36.05}
SYR_POLYGON = {"type": "Polygon", "coordinates": [[
    [SYR_BBOX["lon_min"], SYR_BBOX["lat_min"]],
    [SYR_BBOX["lon_max"], SYR_BBOX["lat_min"]],
    [SYR_BBOX["lon_max"], SYR_BBOX["lat_max"]],
    [SYR_BBOX["lon_min"], SYR_BBOX["lat_max"]],
    [SYR_BBOX["lon_min"], SYR_BBOX["lat_min"]]]]}

LOOKBACK_DAYS = 90          # 7 = last week, 30 = last month, 90 = last quarter
_today = datetime.now(timezone.utc).date()
DATE_END   = _today.isoformat()
DATE_START = (_today - timedelta(days=LOOKBACK_DAYS)).isoformat()
print(f"Area  : Syrian EEZ {SYR_BBOX}")
print(f"Window: {DATE_START} -> {DATE_END}  (last {LOOKBACK_DAYS} days)")
print("Note  : GFW event products lag real time (days-weeks); if you get 0 events, "
      "increase LOOKBACK_DAYS.")

## 5. Syria-flagged vessels (identity registry)

In [ ]:
VESSEL_PAGE = 50   # GFW caps vessels/search "limit" at 50 — larger values 422.

def gfw_vessels_by_flag(flag=SYRIA_FLAG, limit=200):
    entries, since, fetched = [], None, 0
    while fetched < limit:
        params = {"where": f"flag = '{flag}'",
                  "datasets[0]": "public-global-vessel-identity:latest",
                  "limit": min(VESSEL_PAGE, limit - fetched)}
        if since: params["since"] = since
        resp = gfw_get("/vessels/search", params)
        page = (resp or {}).get("entries", []) or []
        if not page: break
        entries.extend(page); fetched += len(page)
        since = (resp or {}).get("since")
        if not since or len(page) < params["limit"]: break
    return {"entries": entries}

def vessels_to_frame(resp):
    rows = []
    for entry in (resp or {}).get("entries", []):
        src = (entry.get("selfReportedInfo") or [{}])[0] or {}
        rg  = (entry.get("registryInfo") or [{}])[0] or {}
        rows.append({
            "vessel_id": src.get("id") or rg.get("id") or entry.get("id"),
            "shipname":  src.get("shipname") or rg.get("shipname"),
            "flag":      src.get("flag") or rg.get("flag"),
            "ssvid_mmsi":src.get("ssvid") or rg.get("ssvid"),
            "imo":       src.get("imo") or rg.get("imo"),
            "callsign":  src.get("callsign") or rg.get("callsign"),
            "geartype":  src.get("geartypes") or rg.get("geartypes"),
        })
    return pd.DataFrame(rows)

syria_vessels = vessels_to_frame(gfw_vessels_by_flag(SYRIA_FLAG, 200))
print(f"Syria-flagged vessels retrieved: {len(syria_vessels)}")
syria_vessels.head(10)

## 6. Events in the Syrian EEZ

Fishing, encounter (rendezvous), loitering, and AIS-gap events inside the EEZ polygon — these are the behaviours IUU investigators care about.

In [ ]:
EVENT_DATASETS = {
    "fishing":   "public-global-fishing-events:latest",
    "encounter": "public-global-encounters-events:latest",
    "loitering": "public-global-loitering-events:latest",
    "gap":       "public-global-gaps-events:latest",
}
EVENT_PAGE = 50    # GFW caps /events "limit" at 50 per page.

def gfw_events_in_region(dataset, start=DATE_START, end=DATE_END, limit=500):
    body = {"datasets": [dataset], "startDate": start, "endDate": end, "geometry": SYR_POLYGON}
    entries, offset, got = [], 0, 0
    while got < limit:
        page_size = min(EVENT_PAGE, limit - got)
        resp = gfw_post("/events", body, params={"limit": page_size, "offset": offset})
        page = (resp or {}).get("entries", []) or []
        if not page: break
        entries.extend(page); got += len(page)
        nxt, total = (resp or {}).get("nextOffset"), (resp or {}).get("total")
        if nxt is None or len(page) < page_size or (total is not None and got >= total): break
        offset = nxt
    return entries

def events_to_frame(entries, etype):
    rows = []
    for ev in entries:
        pos = ev.get("position") or {}; ves = ev.get("vessel") or {}
        rows.append({"type": etype, "event_id": ev.get("id"),
                     "start": ev.get("start"), "end": ev.get("end"),
                     "lat": pos.get("lat"), "lon": pos.get("lon"),
                     "vessel_id": ves.get("id"), "vessel_name": ves.get("name"),
                     "vessel_flag": ves.get("flag"), "ssvid": ves.get("ssvid")})
    return pd.DataFrame(rows)

frames = []
for etype, ds in EVENT_DATASETS.items():
    df = events_to_frame(gfw_events_in_region(ds), etype)
    print(f"  {etype:10s}: {len(df)} events")
    frames.append(df)
syria_events = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"Total Syrian-EEZ events in window: {len(syria_events)}")

if len(syria_events) == 0:
    raise RuntimeError(
        f"No events found in the Syrian EEZ for {DATE_START}..{DATE_END}. GFW event data "
        "lags real time — increase LOOKBACK_DAYS (e.g. 180 or 365) and re-run. "
        "This notebook needs real events to continue (no demo mode)."
    )

syria_events["start"] = pd.to_datetime(syria_events["start"], utc=True, errors="coerce")
syria_events["end"]   = pd.to_datetime(syria_events["end"],   utc=True, errors="coerce")
syria_events.head(10)

## 7. Apparent fishing effort (4Wings report)

In [ ]:
def gfw_fishing_effort(start=DATE_START, end=DATE_END):
    params = {"spatial-resolution": "LOW", "temporal-resolution": "MONTHLY",
              "datasets[0]": "public-global-fishing-effort:latest",
              "date-range": f"{start},{end}", "format": "JSON", "group-by": "FLAG"}
    return gfw_post("/4wings/report", {"geojson": SYR_POLYGON}, params=params)

try:
    syria_effort = gfw_fishing_effort()
    rows = syria_effort.get("entries", [{}])[0]
    key = next(iter(rows)) if rows else None
    effort_df = pd.DataFrame(rows[key]) if key else pd.DataFrame()
    if len(effort_df):
        print("Apparent fishing effort by flag (top 10, hours):")
        print(effort_df.groupby("flag")["hours"].sum().sort_values(ascending=False).head(10).round(1).to_string())
except Exception as e:
    effort_df = pd.DataFrame()
    print("Fishing-effort report unavailable:", e)

## 8. What counts as an *anomaly* here?

"Anomalous" = a vessel whose **behaviour in the Syrian EEZ deviates from the
rest of the fleet** over the same window. We combine two complementary views:

**A. Statistical outliers (unsupervised models).** We summarise each vessel by a
feature vector — event counts (fishing / encounter / loitering / gap), the
*shares* of each, events per active day, spatial footprint, and event durations —
then flag vessels that sit far from the bulk of the fleet:
- **Isolation Forest** — isolates points that are easy to separate (rare profiles).
- **Local Outlier Factor** — flags vessels in low-density regions of feature space.
A vessel is a *model suspect* if **either** model flags it.

**B. Rule-based IUU markers (domain knowledge).** Patterns investigators treat as
red flags regardless of the models:
| Rule | Trigger | Why it matters |
|---|---|---|
| High loitering | ≥ 5 loitering events | staging for transshipment / waiting to meet another vessel |
| Encounter | ≥ 1 at-sea encounter | possible transshipment / rendezvous |
| AIS gap | ≥ 1 "dark" gap event | transponder switched off — hiding activity |
| Unusually busy | events in the top 5% of the fleet | disproportionate activity |

**Final flag — `high_risk` = a vessel that is BOTH a model suspect AND trips ≥ 1
rule.** That intersection keeps the list short and defensible. A continuous
`risk_score` (model scores + rule votes) ranks everyone for triage.

> These are **indicators, not proof** — leads for human review. Legitimate slow
> speed, genuine AIS outages, and anchorage operations all cause false positives.

## 9. Per-vessel feature engineering

In [ ]:
def build_features(ev):
    ev = ev.dropna(subset=["vessel_id"]).copy()
    ev["duration_h"] = (ev["end"] - ev["start"]).dt.total_seconds() / 3600.0
    counts = (ev.pivot_table(index="vessel_id", columns="type", values="event_id",
                             aggfunc="count", fill_value=0)
              .rename(columns=lambda c: f"n_{c}"))
    for col in ("n_fishing", "n_encounter", "n_loitering", "n_gap"):
        if col not in counts.columns: counts[col] = 0
    agg = ev.groupby("vessel_id").agg(
        n_events=("event_id", "count"), vessel_name=("vessel_name", "first"),
        vessel_flag=("vessel_flag", "first"), ssvid=("ssvid", "first"),
        lat_mean=("lat", "mean"), lon_mean=("lon", "mean"),
        lat_std=("lat", "std"), lon_std=("lon", "std"),
        first_seen=("start", "min"), last_seen=("start", "max"),
        total_duration_h=("duration_h", "sum"), mean_duration_h=("duration_h", "mean"))
    agg[["lat_std", "lon_std"]] = agg[["lat_std", "lon_std"]].fillna(0.0)
    agg["active_days"] = ((agg["last_seen"] - agg["first_seen"]).dt.total_seconds() / 86400.0).clip(lower=1.0)
    agg["events_per_day"] = agg["n_events"] / agg["active_days"]
    agg["footprint"] = np.sqrt(agg["lat_std"]**2 + agg["lon_std"]**2)
    feats = agg.join(counts, how="left").fillna(0)
    for share, col in [("loiter_share","n_loitering"), ("enc_share","n_encounter"),
                       ("gap_share","n_gap"), ("fish_share","n_fishing")]:
        feats[share] = feats[col] / feats["n_events"].clip(lower=1)
    return feats.reset_index()

feats = build_features(syria_events)
print(f"Feature matrix: {len(feats)} vessels x {feats.shape[1]} columns")
feats[["vessel_name","vessel_flag","n_events","n_fishing","n_loitering","n_encounter","n_gap"]].head(10)

## 10. Anomaly detection — Isolation Forest + LOF + rules

In [ ]:
MODEL_FEATURES = ["n_events","n_fishing","n_encounter","n_loitering","n_gap",
                  "loiter_share","enc_share","gap_share","events_per_day",
                  "footprint","mean_duration_h","total_duration_h"]

def score(feats):
    X = StandardScaler().fit_transform(feats[MODEL_FEATURES].astype(float).values)
    iso = IsolationForest(n_estimators=300, contamination=0.10, random_state=42, n_jobs=-1).fit(X)
    feats["iso_score"] = -iso.score_samples(X)
    feats["iso_flag"]  = (iso.predict(X) == -1).astype(int)
    n_neigh = min(20, max(5, len(feats)//4))
    lof = LocalOutlierFactor(n_neighbors=n_neigh, contamination=0.10)
    feats["lof_flag"]  = (lof.fit_predict(X) == -1).astype(int)
    feats["lof_score"] = -lof.negative_outlier_factor_
    feats["model_votes"] = feats["iso_flag"] + feats["lof_flag"]
    feats["rule_high_loiter"] = (feats["n_loitering"] >= 5).astype(int)
    feats["rule_encounter"]   = (feats["n_encounter"] >= 1).astype(int)
    feats["rule_ais_gap"]     = (feats["n_gap"] >= 1).astype(int)
    feats["rule_busy"]        = (feats["n_events"] >= feats["n_events"].quantile(0.95)).astype(int)
    feats["rule_votes"] = feats[["rule_high_loiter","rule_encounter","rule_ais_gap","rule_busy"]].sum(axis=1)
    feats["risk_score"] = (feats["iso_score"]/feats["iso_score"].max()
                           + feats["lof_score"]/feats["lof_score"].max()
                           + feats["rule_votes"]/4.0)
    feats["high_risk"]  = ((feats["model_votes"] >= 1) & (feats["rule_votes"] >= 1)).astype(int)
    return feats

feats = score(feats)
feats.to_csv(OUTDIR / "syria_vessel_risk.csv", index=False)
print(f"Isolation Forest flagged : {int(feats.iso_flag.sum())}")
print(f"LOF              flagged : {int(feats.lof_flag.sum())}")
print(f"HIGH-RISK (model+rules)  : {int(feats.high_risk.sum())}")
print(f"Saved per-vessel risk    -> {OUTDIR/'syria_vessel_risk.csv'}")

## 11. Exploratory analysis of the real data

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))

(syria_events["type"].value_counts()
 .reindex(["fishing","loitering","encounter","gap"]).fillna(0)
 .plot.bar(ax=ax[0], color=["#1f77b4","#ff7f0e","#d62728","#000000"]))
ax[0].set(title="Event counts by type", ylabel="events")

ts = syria_events.assign(week=syria_events["start"].dt.to_period("W").dt.start_time)
(ts.groupby(["week","type"]).size().unstack(fill_value=0)
 .plot(ax=ax[1], marker="o", ms=3))
ax[1].set(title="Events over time (weekly)", xlabel="", ylabel="events")

(syria_events["vessel_flag"].fillna("UNKNOWN").value_counts().head(10)
 .plot.barh(ax=ax[2], color="slateblue"))
ax[2].set(title="Top flags by event count"); ax[2].invert_yaxis()
plt.tight_layout(); plt.show()

## 12. Interactive dashboard (filterable layers) + PNG export

Builds an interactive Leaflet/Folium map where each event type and the
high-risk vessels are **separate, toggleable layers** (use the control in the
top-right to filter). Saved to `outputs/syria_dashboard.html`. A static
`outputs/syria_map.png` is also written for reports/slides.

In [ ]:
import folium
from folium.plugins import Fullscreen

EVENT_COLOURS = {"fishing":"#1f77b4","encounter":"#d62728","loitering":"#ff7f0e","gap":"#000000"}
ctr = [(SYR_BBOX["lat_min"]+SYR_BBOX["lat_max"])/2, (SYR_BBOX["lon_min"]+SYR_BBOX["lon_max"])/2]

dashboard = folium.Map(location=ctr, zoom_start=8, tiles="CartoDB positron")
Fullscreen().add_to(dashboard)
folium.Rectangle([(SYR_BBOX["lat_min"], SYR_BBOX["lon_min"]),
                  (SYR_BBOX["lat_max"], SYR_BBOX["lon_max"])],
                 color="green", fill=False, weight=2, tooltip="Syria EEZ (approx)").add_to(dashboard)

# one toggleable layer per event type
for et in ["fishing","loitering","encounter","gap"]:
    fg = folium.FeatureGroup(name=f"{et} events", show=True)
    for _, r in syria_events[(syria_events["type"]==et)].dropna(subset=["lat","lon"]).iterrows():
        folium.CircleMarker([r["lat"], r["lon"]], radius=4, color=EVENT_COLOURS[et],
            fill=True, fill_opacity=0.7,
            tooltip=f"{et} — {r.get('vessel_name')} ({r.get('vessel_flag')})").add_to(fg)
    fg.add_to(dashboard)

# high-risk vessels as a separate layer (at their mean position)
fg = folium.FeatureGroup(name="HIGH-RISK vessels", show=True)
for _, r in feats[feats["high_risk"]==1].iterrows():
    folium.Marker([r["lat_mean"], r["lon_mean"]],
        icon=folium.Icon(color="red", icon="exclamation-sign"),
        tooltip=(f"HIGH RISK: {r['vessel_name']} ({r['vessel_flag']}) | "
                 f"risk={r['risk_score']:.2f} | loiter={int(r['n_loitering'])} "
                 f"enc={int(r['n_encounter'])} gap={int(r['n_gap'])}")).add_to(fg)
fg.add_to(dashboard)

folium.LayerControl(collapsed=False).add_to(dashboard)
dashboard.save(str(OUTDIR / "syria_dashboard.html"))
print(f"Interactive dashboard -> {OUTDIR/'syria_dashboard.html'}")
dashboard

In [ ]:
# Static PNG export (matplotlib — reliable, no browser needed).
import matplotlib.patches as patches
fig, ax = plt.subplots(figsize=(9, 8))
ax.add_patch(patches.Rectangle((SYR_BBOX["lon_min"], SYR_BBOX["lat_min"]),
    SYR_BBOX["lon_max"]-SYR_BBOX["lon_min"], SYR_BBOX["lat_max"]-SYR_BBOX["lat_min"],
    fill=False, ec="green", lw=2))
for et, c in EVENT_COLOURS.items():
    sub = syria_events[syria_events["type"]==et]
    ax.scatter(sub["lon"], sub["lat"], s=18, c=c, label=et, alpha=0.7, edgecolors="none")
hr = feats[feats["high_risk"]==1]
ax.scatter(hr["lon_mean"], hr["lat_mean"], s=180, marker="*", c="red",
           edgecolors="k", label="high-risk vessel", zorder=5)
ax.set(title="Syria EEZ — GFW events & high-risk vessels", xlabel="Longitude", ylabel="Latitude")
ax.legend(loc="upper right"); fig.tight_layout()
fig.savefig(OUTDIR / "syria_map.png", dpi=130); plt.show()
print(f"Static map -> {OUTDIR/'syria_map.png'}")

# Optional: pixel-perfect PNG of the *interactive* map needs a headless browser
# (selenium + chromedriver). Used if available; otherwise the matplotlib PNG above stands.
try:
    (OUTDIR / "syria_dashboard.png").write_bytes(dashboard._to_png(5))
    print(f"Browser-rendered PNG -> {OUTDIR/'syria_dashboard.png'}")
except Exception:
    print("(selenium/chromedriver not available — using the matplotlib PNG as the image export.)")

## 13. Results — the suspect vessels

In [ ]:
cols = ["vessel_name","vessel_flag","ssvid","n_events","n_fishing","n_encounter",
        "n_loitering","n_gap","iso_flag","lof_flag","rule_votes","high_risk","risk_score"]
top = feats.sort_values("risk_score", ascending=False)[cols].head(15)
print("TOP 15 vessels by risk score (Syrian EEZ, current window):")
with pd.option_context("display.width", 200, "display.precision", 3):
    print(top.to_string(index=False))

## 15. Enrichment layers — *flagged vessels only*

Everything above runs on the full fleet. From here we take **only the flagged /
suspicious vessels** (the model + rule suspects — ~hundreds in a typical run) and
build a dossier on each by layering four extra sources:

1. **VesselAPI** — live position / identity *(free plan ≈150 calls/mo → capped)*
2. **DataDocked** — vessel particulars by MMSI/IMO *(bulk-capable)*
3. **OFAC sanctions** — free SDN screening (one download, no token)
4. **Equasis** — registered owner / manager / class *(no public API → merge a CSV
   you export; ToS-respecting)*

**Design: detect broad, enrich narrow.** Free/bulk layers (OFAC, DataDocked) run
on *all* suspects; the metered VesselAPI runs on a small Top-N. Every response is
**cached to disk**, so re-running enrichment costs **0** extra API calls.

In [ ]:
# --- select the flagged/suspicious subset from the scored feature table ------
suspects = feats[(feats["high_risk"] == 1) | (feats["model_votes"] >= 1)].copy()
suspects = suspects.sort_values("risk_score", ascending=False).reset_index(drop=True)
# MMSI is our join key into the other databases (GFW 'ssvid' == MMSI).
suspects["mmsi"] = (suspects["ssvid"].astype(str)
                    .str.replace(r"\.0$", "", regex=True).replace("nan", ""))
suspects.to_csv(OUTDIR / "syria_suspects.csv", index=False)
print(f"Flagged/suspicious vessels to enrich: {len(suspects)} "
      f"(high_risk={int(feats['high_risk'].sum())}, model-suspects={int((feats['model_votes']>=1).sum())})")
print(f"With a usable MMSI: {int((suspects['mmsi'].str.len() > 0).sum())}")
suspects[["vessel_name","vessel_flag","mmsi","n_events","risk_score","high_risk"]].head(10)

### Hidden API tokens for the enrichment layers

Each prompt is a **masked input** (nothing is echoed). Leave a prompt blank to
**skip** that layer. (Your GFW token in Section 2 is already a hidden prompt.)

In [ ]:
import os, time, hashlib, getpass

def hidden_token(label):
    try:
        return getpass.getpass(f"{label} (leave blank to skip this layer): ").strip()
    except Exception:
        return ""   # non-interactive (e.g. nbconvert) -> skipped

VESSELAPI_TOKEN  = os.environ.get("VESSELAPI_TOKEN", "")  or hidden_token("VesselAPI token")
DATADOCKED_TOKEN = os.environ.get("DATADOCKED_TOKEN", "") or hidden_token("DataDocked token")
# Equasis has no public API (see Layer 4). These are only used if you later wire
# in authorized access; they are NOT used to scrape the website.
EQUASIS_USER = os.environ.get("EQUASIS_USER", "")
EQUASIS_PASS = os.environ.get("EQUASIS_PASS", "")

print("Layers enabled:",
      "VesselAPI" if VESSELAPI_TOKEN else "—",
      "| DataDocked" if DATADOCKED_TOKEN else "| —",
      "| OFAC (always, free)")

# --- frugal cached client (per-API cache + monthly budget + per-run cap) ------
ENRICH_CACHE = OUTDIR / "enrich_cache"; ENRICH_CACHE.mkdir(exist_ok=True)
def make_client(name, base, headers, monthly_budget, run_budget):
    usage = ENRICH_CACHE / f"{name}_usage.json"; run = [0]
    def _u():
        if usage.exists():
            u = json.loads(usage.read_text())
            if u.get("month") == time.strftime("%Y-%m"): return u
        return {"month": time.strftime("%Y-%m"), "count": 0}
    def get(path, params, label=""):
        key = hashlib.sha1((name+path+json.dumps(params, sort_keys=True)).encode()).hexdigest()[:16]
        cf = ENRICH_CACHE / f"{name}_{key}.json"
        if cf.exists(): return json.loads(cf.read_text())
        u = _u()
        if u["count"] >= monthly_budget: raise RuntimeError(f"{name}: monthly budget {monthly_budget} reached.")
        if run[0] >= run_budget: raise RuntimeError(f"{name}: per-run cap {run_budget} reached.")
        r = requests.get(f"{base}{path}", headers=headers, params=params, timeout=60)
        try: r.raise_for_status()
        except requests.HTTPError as e: raise requests.HTTPError(f"{e} | {r.text[:300]}") from None
        resp = r.json(); run[0]+=1; u["count"]+=1; usage.write_text(json.dumps(u)); cf.write_text(json.dumps(resp))
        return resp
    return get

### Layer 2: DataDocked — vessel particulars by MMSI (runs on ALL suspects)

In [ ]:
ENRICH_MAX_DD = len(suspects)          # bulk plan; lower this to spend fewer calls
dd_get = (make_client("datadocked", "https://datadocked.com/api",
                      {"accept": "application/json", "x-api-key": DATADOCKED_TOKEN},
                      monthly_budget=1000, run_budget=ENRICH_MAX_DD + 5)
          if DATADOCKED_TOKEN else None)

def dd_info(mmsi):
    if not dd_get or not mmsi: return {}
    try:
        resp = dd_get("/vessels_operations/get-vessel-info", {"imo_or_mmsi": mmsi}, label=f"DD {mmsi}")
    except RuntimeError as e:
        return {"_stop": str(e)}
    except Exception as e:
        return {"_err": str(e)}
    d = resp.get("data") if isinstance(resp, dict) else None
    v = d if isinstance(d, dict) else (resp if isinstance(resp, dict) else {})
    return {"imo": v.get("imo"), "dd_type": v.get("type") or v.get("vesselType"),
            "dd_flag": v.get("flag") or v.get("country_iso"),
            "dd_owner": v.get("owner"), "dd_built": v.get("yearBuilt") or v.get("year_built"),
            "dd_length": v.get("length") or v.get("length_m")}

rows = []
if dd_get:
    for i, r in suspects.head(ENRICH_MAX_DD).iterrows():
        info = dd_info(r["mmsi"])
        if info.get("_stop"): print("DataDocked stopped:", info["_stop"]); break
        rows.append({"vessel_id": r["vessel_id"], **{k: v for k, v in info.items() if not k.startswith("_")}})
    print(f"DataDocked enriched {len(rows)} vessels.")
else:
    print("DataDocked layer skipped (no token).")
dd_df = pd.DataFrame(rows) if rows else pd.DataFrame(columns=["vessel_id"])

### Layer 1: VesselAPI — live identity by name (capped Top-N; free plan ≈150/mo)

In [ ]:
ENRICH_MAX_VAPI = 25                    # keep small: free plan is ~150 calls/MONTH
va_get = (make_client("vesselapi", "https://api.vesselapi.com/v1",
                      {"Authorization": f"Bearer {VESSELAPI_TOKEN}"},
                      monthly_budget=150, run_budget=ENRICH_MAX_VAPI + 2)
          if VESSELAPI_TOKEN else None)

def va_lookup(name):
    if not va_get or not name: return {}
    try:
        resp = va_get("/search/vessels", {"filterName": name}, label=f"VA {name}")
    except RuntimeError as e:
        return {"_stop": str(e)}
    except Exception as e:
        return {"_err": str(e)}
    arr = (resp or {}).get("vessels", []) or []
    v = arr[0] if arr else {}
    return {"va_imo": v.get("imo"), "va_mmsi": v.get("mmsi"),
            "va_type": v.get("vesselType"), "va_callsign": v.get("callSign")}

rows = []
if va_get:
    for i, r in suspects.head(ENRICH_MAX_VAPI).iterrows():
        info = va_lookup(r["vessel_name"])
        if info.get("_stop"): print("VesselAPI stopped:", info["_stop"]); break
        rows.append({"vessel_id": r["vessel_id"], **{k: v for k, v in info.items() if not k.startswith("_")}})
    print(f"VesselAPI enriched {len(rows)} vessels (cap {ENRICH_MAX_VAPI}).")
else:
    print("VesselAPI layer skipped (no token).")
va_df = pd.DataFrame(rows) if rows else pd.DataFrame(columns=["vessel_id"])

### Layer 3: OFAC sanctions screening (free — one download, no token)

Downloads the OFAC **SDN list**, keeps the *vessel* entries, extracts each
sanctioned vessel's IMO (from the remarks) and name, and screens our suspects by
**IMO** (from Layers 1/2) and by **normalised name**.

In [ ]:
import io, re
SDN_URL = "https://www.treasury.gov/ofac/downloads/sdn.csv"   # public OFAC SDN list
SDN_COLS = ["ent_num","name","sdn_type","program","title","callsign",
            "vess_type","tonnage","grt","vess_flag","vess_owner","remarks"]
sdn_cache = ENRICH_CACHE / "ofac_sdn.csv"

try:
    if not sdn_cache.exists():
        sdn_cache.write_bytes(requests.get(SDN_URL, timeout=60).content)
    raw = sdn_cache.read_text(errors="ignore")
    sdn = pd.read_csv(io.StringIO(raw), names=SDN_COLS, dtype=str, quotechar='"',
                      on_bad_lines="skip").replace("-0-", np.nan)
    sv = sdn[sdn["sdn_type"].astype(str).str.lower() == "vessel"].copy()
    sv["imo"] = sv["remarks"].str.extract(r"IMO\s*(\d{7})", flags=re.I)
    sv["name_norm"] = sv["name"].str.upper().str.strip()
    sanc_imo = set(sv["imo"].dropna()); sanc_names = set(sv["name_norm"].dropna())
    prog_by_name = sv.dropna(subset=["name_norm"]).set_index("name_norm")["program"].to_dict()
    print(f"OFAC SDN: {len(sv)} sanctioned vessels ({len(sanc_imo)} with IMO).")
except Exception as e:
    print("OFAC download/parse failed (continuing without it):", e)
    sanc_imo, sanc_names, prog_by_name = set(), set(), {}

def screen(row, imo):
    nm = str(row.get("vessel_name", "")).upper().strip()
    hit_name = nm in sanc_names
    hit_imo  = bool(imo) and str(imo) in sanc_imo
    return pd.Series({"sanctioned": bool(hit_name or hit_imo),
                      "ofac_program": prog_by_name.get(nm)})

### Layer 4: Equasis — owner / manager / class  *(manual, ToS-respecting)*

[Equasis](https://www.equasis.org) has **no public API**, and its terms restrict
automated extraction — so this notebook does **not** scrape it. Instead it writes
the flagged **IMOs to a to-do CSV**; you look them up while logged in to Equasis
(or use an authorized data feed), save the result as
`outputs/equasis_filled.csv` with columns
`imo, registered_owner, manager, class_society, pi_club`, and **re-run** this cell
to merge.

In [ ]:
# We need IMOs first (from DataDocked/VesselAPI). Build a working IMO column.
def best_imo(vid):
    for d, col in [(dd_df, "imo"), (va_df, "va_imo")]:
        if "vessel_id" in d and col in d:
            m = d.loc[d["vessel_id"] == vid, col]
            if len(m) and pd.notnull(m.iloc[0]) and str(m.iloc[0]) not in ("", "0", "nan"):
                return str(m.iloc[0]).split(".")[0]
    return None
suspects["imo"] = suspects["vessel_id"].map(best_imo)

todo = suspects.dropna(subset=["imo"])[["imo", "vessel_name", "vessel_flag"]].drop_duplicates()
todo.to_csv(OUTDIR / "equasis_lookup_TODO.csv", index=False)

eq_path = OUTDIR / "equasis_filled.csv"
if eq_path.exists():
    eq = pd.read_csv(eq_path, dtype=str)
    print(f"Merging Equasis details for {eq['imo'].nunique()} vessels from {eq_path}.")
else:
    eq = pd.DataFrame(columns=["imo","registered_owner","manager","class_society","pi_club"])
    print(f"Wrote {len(todo)} IMOs -> {OUTDIR/'equasis_lookup_TODO.csv'}. "
          "Look them up on Equasis, save outputs/equasis_filled.csv "
          "(imo,registered_owner,manager,class_society,pi_club), then re-run this cell.")

### Combine all layers + enriched re-ranking

In [ ]:
enriched = suspects.copy()
for d in (dd_df, va_df):
    if len(d) and "vessel_id" in d:
        enriched = enriched.merge(d, on="vessel_id", how="left", suffixes=("", "_dup"))
# OFAC screening (uses the IMO we resolved above)
scr = enriched.apply(lambda r: screen(r, r.get("imo")), axis=1)
enriched = pd.concat([enriched, scr], axis=1)
# Equasis merge (if provided)
if len(eq):
    enriched = enriched.merge(eq, on="imo", how="left")

# Enriched risk: sanctions dominate; identity gaps and age add a little.
enriched["enriched_risk"] = (
    enriched["risk_score"].astype(float)
    + enriched["sanctioned"].fillna(False).astype(int) * 5.0
    + enriched.get("imo", pd.Series(index=enriched.index, dtype=object)).isna().astype(int) * 0.5
)
enriched = enriched.sort_values("enriched_risk", ascending=False).reset_index(drop=True)
enriched.to_csv(OUTDIR / "syria_enriched_suspects.csv", index=False)

show = [c for c in ["vessel_name","vessel_flag","mmsi","imo","dd_type","dd_owner",
                    "registered_owner","sanctioned","ofac_program","n_events",
                    "risk_score","enriched_risk"] if c in enriched.columns]
print(f"Enriched {len(enriched)} suspects | sanctioned hits: {int(enriched['sanctioned'].sum())}")
print(f"Saved -> {OUTDIR/'syria_enriched_suspects.csv'}")
with pd.option_context("display.width", 220, "display.max_columns", 40):
    print(enriched[show].head(15).to_string(index=False))

### Enriched dashboard — suspects with all layers

In [ ]:
import folium
from folium.plugins import Fullscreen
ctr = [(SYR_BBOX["lat_min"]+SYR_BBOX["lat_max"])/2, (SYR_BBOX["lon_min"]+SYR_BBOX["lon_max"])/2]
em = folium.Map(location=ctr, zoom_start=8, tiles="CartoDB positron")
Fullscreen().add_to(em)
folium.Rectangle([(SYR_BBOX["lat_min"], SYR_BBOX["lon_min"]),
                  (SYR_BBOX["lat_max"], SYR_BBOX["lon_max"])],
                 color="green", fill=False, weight=2, tooltip="Syria EEZ").add_to(em)

def layer_for(r):
    if r.get("sanctioned"): return "SANCTIONED (OFAC)"
    if r.get("high_risk") == 1: return "High-risk"
    return "Other suspects"
LCOL = {"SANCTIONED (OFAC)": "#000000", "High-risk": "#d62728", "Other suspects": "#ff7f0e"}
enriched["_layer"] = enriched.apply(layer_for, axis=1)
for lay, col in LCOL.items():
    fg = folium.FeatureGroup(name=lay, show=True)
    for _, r in enriched[enriched["_layer"] == lay].dropna(subset=["lat_mean","lon_mean"]).iterrows():
        tip = (f"{r['vessel_name']} ({r.get('vessel_flag')}) | MMSI {r['mmsi']} | IMO {r.get('imo')}<br>"
               f"owner: {r.get('dd_owner') or r.get('registered_owner') or '?'}<br>"
               f"sanctioned: {bool(r.get('sanctioned'))} ({r.get('ofac_program')}) | "
               f"risk {r.get('enriched_risk'):.2f}")
        folium.CircleMarker([r["lat_mean"], r["lon_mean"]],
            radius=8 if lay.startswith("SANC") else 6, color=col, fill=True, fill_opacity=0.85,
            tooltip=tip).add_to(fg)
    fg.add_to(em)
folium.LayerControl(collapsed=False).add_to(em)
em.save(str(OUTDIR / "syria_enriched_dashboard.html"))
print(f"Enriched dashboard -> {OUTDIR/'syria_enriched_dashboard.html'}")
em

## 16. Enrichment — limits & notes

- **Budget reality.** Enriching ~220 vessels via **VesselAPI's free 150-call/month**
  plan isn't possible, so VesselAPI is capped (`ENRICH_MAX_VAPI`, default 25) while
  the **free** OFAC layer and the **bulk** DataDocked layer cover all suspects.
  The disk cache means re-runs spend **0** calls.
- **Join keys.** GFW events give MMSI (`ssvid`); IMO is resolved from
  DataDocked/VesselAPI and then used for OFAC-by-IMO and Equasis.
- **OFAC** screening is by IMO + normalised name (a *screening aid*, not legal
  advice — confirm hits against the official SDN entry). Consider
  [OpenSanctions](https://www.opensanctions.org) for fuzzy matching / EU/UN lists.
- **Equasis** is intentionally a manual CSV merge (no scraping; respects ToS).
- **Outputs:** `syria_suspects.csv`, `syria_enriched_suspects.csv`,
  `equasis_lookup_TODO.csv`, `syria_enriched_dashboard.html`.

## 17. How to continue from here

A practical roadmap, roughly in priority order:

1. **Tighten the AOI.** Swap the bounding box for the official **Syria EEZ
   polygon** (Marine Regions / GFW `public-eez-areas`) so events on land or in
   neighbouring EEZs don't leak in.
2. **Lengthen the window for baselining.** Pull 6–12 months to learn each
   vessel's *normal* rhythm, then score the **recent** week/month against that
   personal baseline (deviation-from-self, not just deviation-from-fleet).
3. **Enrich with GFW Insights & identity.** Join the per-vessel **Insights**
   endpoint (gear, registry, prior violations) and flag identity red flags
   (flag-hopping, no IMO, recent re-flagging).
4. **Model the encounters directly.** For each `encounter`, pull both vessels and
   check whether one is a known reefer/carrier — the classic transshipment
   signature — rather than scoring vessels in isolation.
5. **Add AIS track features.** The current model is event-level; pulling raw
   positions enables speed/turn/gap/jump features per track and sequence models
   (LSTM/transformer) for behaviour-change detection.
6. **Calibrate thresholds with feedback.** Capture analyst "true/false lead"
   verdicts and tune contamination + rule thresholds to a target precision.
7. **Operationalise.** Schedule a daily pull, write `outputs/` to object storage,
   and serve the dashboard + risk table to analysts; alert on new high-risk vessels.